In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
import cv2
from sklearn.model_selection import train_test_split
import pickle
import os
import pandas as pd
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
!gdown "https://drive.google.com/uc?id=1Tzw4hHHRIhkJpCeFye5kafD_Go7A4vNv&export=download" -O dataset.zip


Downloading...
From (original): https://drive.google.com/uc?id=1Tzw4hHHRIhkJpCeFye5kafD_Go7A4vNv&export=download
From (redirected): https://drive.google.com/uc?id=1Tzw4hHHRIhkJpCeFye5kafD_Go7A4vNv&export=download&confirm=t&uuid=1f8ab647-1cda-41b3-9e72-659738f2a861
To: /content/dataset.zip
100% 84.7M/84.7M [00:00<00:00, 110MB/s]


In [3]:
import zipfile

try:
    with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
        zip_ref.extractall()
    print("Dataset successfully extracted!")
except zipfile.BadZipFile:
    print("The file is not a valid zip file.")


Dataset successfully extracted!


In [4]:
path = "Dataset"
labelFile = 'labels.csv'
batch_size_val=32
epochs_val=10
imageDimesions = (32,32,3)
testRatio = 0.2
validationRatio = 0.2

In [5]:
count = 0
images = []
classNo = []
myList = os.listdir(path)
print("Total Classes Detected:",len(myList))
noOfClasses=len(myList)
print("Importing Classes.....")
for x in range (0,len(myList)):
    myPicList = os.listdir(path+"/"+str(count))
    for y in myPicList:
        curImg = cv2.imread(path+"/"+str(count)+"/"+y)
        images.append(curImg)
        classNo.append(count)
    print(count, end =" ")
    count +=1
print(" ")
images = np.array(images)
classNo = np.array(classNo)

Total Classes Detected: 43
Importing Classes.....
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42  


In [6]:
X_train, X_test, y_train, y_test = train_test_split(images, classNo, test_size=testRatio)
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train, test_size=validationRatio)

In [7]:
print("Data Shapes")
print("Train",end = "");print(X_train.shape,y_train.shape)
print("Validation",end = "");print(X_validation.shape,y_validation.shape)
print("Test",end = "");print(X_test.shape,y_test.shape)

Data Shapes
Train(22271, 32, 32, 3) (22271,)
Validation(5568, 32, 32, 3) (5568,)
Test(6960, 32, 32, 3) (6960,)


In [8]:
data=pd.read_csv(labelFile)
print("data shape ",data.shape,type(data))

num_of_samples = []
cols = 5
num_classes = noOfClasses

def grayscale(img):
    img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    return img
def equalize(img):
    img =cv2.equalizeHist(img)
    return img
def preprocessing(img):
    img = grayscale(img)
    img = equalize(img)
    img = img/255
    return img

X_train=np.array(list(map(preprocessing,X_train)))
X_validation=np.array(list(map(preprocessing,X_validation)))
X_test=np.array(list(map(preprocessing,X_test)))

data shape  (43, 2) <class 'pandas.core.frame.DataFrame'>


In [9]:
X_train=X_train.reshape(X_train.shape[0],X_train.shape[1],X_train.shape[2],1)
X_validation=X_validation.reshape(X_validation.shape[0],X_validation.shape[1],X_validation.shape[2],1)
X_test=X_test.reshape(X_test.shape[0],X_test.shape[1],X_test.shape[2],1)


dataGen= ImageDataGenerator(width_shift_range=0.1,
                            height_shift_range=0.1,
                            zoom_range=0.2,
                            shear_range=0.1,
                            rotation_range=10)
dataGen.fit(X_train)
batches= dataGen.flow(X_train,y_train,batch_size=20)
X_batch,y_batch = next(batches)


y_train = to_categorical(y_train,noOfClasses)
y_validation = to_categorical(y_validation,noOfClasses)
y_test = to_categorical(y_test,noOfClasses)

In [10]:
from tensorflow.keras.optimizers import Adam

def myModel():
    model = Sequential()
    model.add(Conv2D(60, (5, 5), input_shape=(imageDimesions[0], imageDimesions[1], 1), activation='relu'))
    model.add(Conv2D(60, (5, 5), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(30, (3, 3), activation='relu'))
    model.add(Conv2D(30, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.5))

    model.add(Flatten())
    model.add(Dense(500, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(noOfClasses, activation='softmax'))

    # Use `learning_rate` instead of `lr`
    model.compile(Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [11]:
model = myModel()

# Display the model summary
print(model.summary())

# Train the model
history = model.fit(
    dataGen.flow(X_train, y_train, batch_size=batch_size_val),
    steps_per_epoch=len(X_train) // batch_size_val,
    epochs=epochs_val,
    validation_data=(X_validation, y_validation),
    shuffle=True
)

# Plot loss and accuracy
plt.figure(1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['training', 'validation'])
plt.title('Loss')
plt.xlabel('Epoch')

plt.figure(2)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['training', 'validation'])
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.show()

# Evaluate the model
score = model.evaluate(X_test, y_test, verbose=0)
print('Test Score:', score[0])
print('Test Accuracy:', score[1])

# Save the model
model.save("model.h5")

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 28, 28, 60)          │           1,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 24, 24, 60)          │          90,060 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 12, 12, 60)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 10, 10, 30)          │          16,230 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 8, 8, 30)            │           8,130 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 4, 4, 30)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 4, 4, 30)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 480)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 500)                 │         240,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 500)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 43)                  │          21,543 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 378,023 (1.44 MB)

 Trainable params: 378,023 (1.44 MB)

...

/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


695/695 ━━━━━━━━━━━━━━━━━━━━ 287s 407ms/step - accuracy: 0.1850 - loss: 3.0035 - val_accuracy: 0.8080 - val_loss: 0.6568
Epoch 2/10
  1/695 ━━━━━━━━━━━━━━━━━━━━ 3:44 324ms/step - accuracy: 0.5312 - loss: 1.3432

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


695/695 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - accuracy: 0.5312 - loss: 1.3432 - val_accuracy: 0.8222 - val_loss: 0.6395
Epoch 3/10
695/695 ━━━━━━━━━━━━━━━━━━━━ 305s 406ms/step - accuracy: 0.6254 - loss: 1.2051 - val_accuracy: 0.9477 - val_loss: 0.1946
Epoch 4/10
695/695 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - accuracy: 0.7500 - loss: 0.6779 - val_accuracy: 0.9479 - val_loss: 0.1894
Epoch 5/10
695/695 ━━━━━━━━━━━━━━━━━━━━ 294s 391ms/step - accuracy: 0.7657 - loss: 0.7562 - val_accuracy: 0.9587 - val_loss: 0.1440
Epoch 6/10
695/695 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - accuracy: 0.7500 - loss: 0.7282 - val_accuracy: 0.9626 - val_loss: 0.1374
Epoch 7/10
585/695 ━━━━━━━━━━━━━━━━━━━━ 41s 375ms/step - accuracy: 0.8136 - loss: 0.5922

KeyboardInterrupt: 

In [12]:
import ipywidgets as widgets

In [13]:
import numpy as np
import cv2
import tensorflow as tf
from google.colab import files
import ipywidgets as widgets
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

# Define image dimensions (should match what the model expects)
imageDimesions = (32, 32, 3)

# Load your pre-trained model
model = load_model("model.h5")  # Load the saved model

# Define a list of class labels (modify according to your dataset)
# Here, we assume that class 0 corresponds to "traffic sign" and other classes do not
class_labels = {
    0: "Traffic Sign",
    1: "Not a Traffic Sign"  # You can add more mappings if needed
}

# Create a file upload widget
upload_widget = widgets.FileUpload(
    accept='image/*',  # Restrict file types to images
    multiple=False  # Allow only one file to be uploaded at a time
)

# Display the widget
display(upload_widget)

# Preprocess image function
def preprocess_image(image, image_dimensions):
    # Decode image bytes to image
    image = cv2.imdecode(np.frombuffer(image, np.uint8), -1)

    # Convert the image to grayscale if it's RGB
    if len(image.shape) == 3 and image.shape[2] == 3:  # RGB image
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Resize image to match the input size of the model
    image = cv2.resize(image, (image_dimensions[0], image_dimensions[1]))

    # Normalize the image
    image = image / 255.0

    # Reshape the image to the format the model expects (batch size, width, height, channels)
    image = image.reshape(1, image_dimensions[0], image_dimensions[1], 1)  # Grayscale image

    return image

# Predict the class of the image
def predict_image(image_data):
    preprocessed_image = preprocess_image(image_data, imageDimesions)
    predictions = model.predict(preprocessed_image)
    predicted_class = np.argmax(predictions, axis=1)[0]  # Get the predicted class
    confidence = np.max(predictions)  # Get the confidence score
    return predicted_class, confidence

# Function to handle file upload and prediction
def on_upload_change(change):
    if upload_widget.value:
        uploaded_image = list(upload_widget.value.values())[0]['content']  # Get the uploaded image
        predicted_class, confidence = predict_image(uploaded_image)  # Call the model to predict

        # Get the label based on the predicted class
        label = class_labels.get(predicted_class, "Unknown Class")

        # If it's a traffic sign (class 0), show message accordingly
        if predicted_class == 0:
            print(f"Predicted: Traffic Sign (Class {predicted_class})")
        else:
            print(f"Predicted: Not a Traffic Sign (Class {predicted_class})")

        print(f"Confidence: {confidence:.2f}")

        # Display the uploaded image
        uploaded_image = cv2.imdecode(np.frombuffer(uploaded_image, np.uint8), -1)
        plt.imshow(uploaded_image, cmap='gray')
        plt.title(f"Predicted: {label} (Confidence: {confidence:.2f})")
        plt.axis("off")
        plt.show()

# Attach the function to the upload widget
upload_widget.observe(on_upload_change, names='value')


OSError: Unable to synchronously open file (truncated file: eof = 1048576, sblock->base_addr = 0, stored_eof = 4593640)

In [17]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model

# Define image dimensions (should match what the model expects)
imageDimensions = (32, 32, 3)  # Make sure this matches your model's input size

# Load your pre-trained model
model = load_model("model.h5")  # Load the saved model

# Define class labels (0 = Traffic Sign, 1 = Not a Traffic Sign)
class_labels = {0: "Traffic Sign", 1: "Not a Traffic Sign"}

# Preprocess image function
def preprocess_image(image, image_dimensions):
    # Convert the image to grayscale if it's RGB (to match your original preprocessing)
    if len(image.shape) == 3 and image.shape[2] == 3:  # RGB image
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Resize the image to match the input size of the model
    image = cv2.resize(image, (image_dimensions[0], image_dimensions[1]))

    # Normalize the image
    image = image / 255.0

    # Reshape the image to the format the model expects (batch size, width, height, channels)
    image = image.reshape(1, image_dimensions[0], image_dimensions[1], 1)  # Grayscale image

    return image

# Predict the class of the image
def predict_image(image_data):
    preprocessed_image = preprocess_image(image_data, imageDimensions)
    predictions = model.predict(preprocessed_image)
    predicted_class = np.argmax(predictions, axis=1)[0]  # Get the predicted class
    confidence = np.max(predictions)  # Get the confidence score
    return predicted_class, confidence

# Start video capture (open the webcam)
cap = cv2.VideoCapture(0)  # Use 0 for the default webcam, or change it if you have a different camera

while True:
    # Capture frame-by-frame from the camera
    ret, frame = cap.read()

    if not ret:
        break  # If no frame is captured, exit

    # Predict the class of the current frame
    predicted_class, confidence = predict_image(frame)

    # Get the label based on the predicted class
    label = class_labels.get(predicted_class, "Unknown Class")

    # Display the result
    print(f"Predicted: {label} (Confidence: {confidence:.2f})")

    # Display the frame with the prediction result
    cv2.putText(frame, f"Predicted: {label} (Confidence: {confidence:.2f})",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)

    # Show the frame with prediction
    cv2.imshow("Camera Feed", frame)

    # If 'q' is pressed, close the camera feed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture object and close any open windows
cap.release()
cv2.destroyAllWindows()


In [15]:
pip install opencv-python


In [ ]:
cap = cv2.VideoCapture(9)  # Use 0 for the default webcam

if not cap.isOpened():
    print("Error: Could not open webcam.")
else:
    print("Webcam opened successfully.")


Error: Could not open webcam.


In [ ]:
# from IPython.display import display, Javascript, Image
# import cv2
# import numpy as np
# import tensorflow as tf
# from tensorflow.keras.models import load_model
# import ipywidgets as widgets
# from base64 import b64decode
# from io import BytesIO
# from PIL import Image as PILImage

# # Load the pre-trained model
# model = load_model("model.h5")

# # Define image dimensions
# imageDimensions = (32, 32, 3)

# # Preprocess the image
# def preprocess_image(image, image_dimensions):
#     # Convert the image to grayscale
#     if len(image.shape) == 3 and image.shape[2] == 3:  # RGB image
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

#     # Resize the image to the required dimensions
#     image = cv2.resize(image, (image_dimensions[0], image_dimensions[1]))

#     # Normalize the image
#     image = image / 255.0

#     # Reshape for model
#     image = image.reshape(1, image_dimensions[0], image_dimensions[1], 1)

#     return image

# # Predict function
# def predict_image(image_data):
#     preprocessed_image = preprocess_image(image_data, imageDimensions)
#     predictions = model.predict(preprocessed_image)
#     predicted_class = np.argmax(predictions, axis=1)[0]
#     confidence = np.max(predictions)
#     return predicted_class, confidence

# # JavaScript to capture image from webcam
# def capture_image_from_webcam():
#     display(Javascript('''
#         const video = document.createElement('video');
#         video.width = 640;
#         video.height = 480;
#         video.autoplay = true;

#         // Get access to the webcam
#         navigator.mediaDevices.getUserMedia({video: true})
#             .then(stream => {
#                 video.srcObject = stream;
#                 document.body.appendChild(video);

#                 // Capture the image when video is playing
#                 video.onplaying = function() {
#                     setTimeout(function() {
#                         const canvas = document.createElement('canvas');
#                         canvas.width = 640;
#                         canvas.height = 480;
#                         canvas.getContext('2d').drawImage(video, 0, 0);
#                         const imageData = canvas.toDataURL('image/jpeg');
#                         document.body.removeChild(video);
#                         google.colab.kernel.invokeFunction('notebook.capture_image', [imageData], {});
#                     }, 2000);
#                 };
#             })
#             .catch(err => {
#                 alert('Error: ' + err);
#             });
#     '''))

# # Callback to handle the captured image
# def handle_captured_image(image_data):
#     # Decode the base64 image
#     image_data = image_data.split(',')[1]  # Remove the "data:image/jpeg;base64," part
#     image_data = b64decode(image_data)

#     # Convert the byte data to an image
#     image = PILImage.open(BytesIO(image_data))
#     image = np.array(image)

#     # Predict using the model
#     predicted_class, confidence = predict_image(image)

#     print(f"Predicted Class: {predicted_class}")
#     print(f"Confidence: {confidence:.2f}")

#     # Show the captured image and prediction
#     plt.imshow(image)
#     plt.title(f"Predicted Class: {predicted_class} (Confidence: {confidence:.2f})")
#     plt.axis('off')
#     plt.show()

# # Register the Python callback with Colab
# from google.colab import output
# output.register_callback('notebook.capture_image', handle_captured_image)

# # Start the webcam capture
# capture_image_from_webcam()


In [18]:
# from IPython.display import display, Javascript, Image
# import cv2
# import numpy as np
# import tensorflow as tf
# from tensorflow.keras.models import load_model
# import ipywidgets as widgets
# from base64 import b64decode
# from io import BytesIO
# from PIL import Image as PILImage

# # Load the pre-trained model
# model = load_model("model.h5")

# # Define image dimensions
# imageDimensions = (32, 32, 3)

# # Preprocess the image
# def preprocess_image(image, image_dimensions):
#     # Convert the image to grayscale
#     if len(image.shape) == 3 and image.shape[2] == 3:  # RGB image
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

#     # Resize the image to the required dimensions
#     image = cv2.resize(image, (image_dimensions[0], image_dimensions[1]))

#     # Normalize the image
#     image = image / 255.0

#     # Reshape for model
#     image = image.reshape(1, image_dimensions[0], image_dimensions[1], 1)

#     return image

# # Predict function
# def predict_image(image_data):
#     preprocessed_image = preprocess_image(image_data, imageDimensions)
#     predictions = model.predict(preprocessed_image)
#     predicted_class = np.argmax(predictions, axis=1)[0]
#     confidence = np.max(predictions)
#     return predicted_class, confidence

# # JavaScript to capture image from webcam and display confidence level
# def capture_image_from_webcam():
#     display(Javascript('''
#         const video = document.createElement('video');
#         video.width = 640;
#         video.height = 480;
#         video.autoplay = true;

#         // Get access to the webcam
#         navigator.mediaDevices.getUserMedia({video: true})
#             .then(stream => {
#                 video.srcObject = stream;
#                 document.body.appendChild(video);

#                 // Create a canvas to overlay graphics
#                 const canvas = document.createElement('canvas');
#                 canvas.width = 640;
#                 canvas.height = 480;
#                 const context = canvas.getContext('2d');
#                 document.body.appendChild(canvas);

#                 // Capture and predict every frame
#                 function captureAndPredict() {
#                     context.drawImage(video, 0, 0, 640, 480);
#                     const imageData = canvas.toDataURL('image/jpeg');
#                     google.colab.kernel.invokeFunction('notebook.capture_image', [imageData], {});
#                     requestAnimationFrame(captureAndPredict);
#                 }

#                 captureAndPredict();
#             })
#             .catch(err => {
#                 alert('Error: ' + err);
#             });
#     '''))

# # Callback to handle the captured image and display predictions
# def handle_captured_image(image_data):
#     # Decode the base64 image
#     image_data = image_data.split(',')[1]  # Remove the "data:image/jpeg;base64," part
#     image_data = b64decode(image_data)

#     # Convert the byte data to an image
#     image = PILImage.open(BytesIO(image_data))
#     image = np.array(image)

#     # Predict using the model
#     predicted_class, confidence = predict_image(image)

#     # Display prediction and confidence
#     if confidence > 0.5:  # Confidence greater than 50%
#         # Draw green square
#         context = document.querySelector('canvas').getContext('2d');
#         context.beginPath();
#         context.rect(10, 10, 50, 50);
#         context.fillStyle = 'green';
#         context.fill();
#         context.stroke();


#     print(f"Predicted Class: {predicted_class}")
#     print(f"Confidence: {confidence:.2f}")

# # Register the Python callback with Colab
# from google.colab import output
# output.register_callback('notebook.capture_image', handle_captured_image)

# # Start the webcam capture
# capture_image_from_webcam()


<IPython.core.display.Javascript object>

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
Predicted Class: 32
Confidence: 0.20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Predicted Class: 25
Confidence: 0.34
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Predicted Class: 25
Confidence: 0.19
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Predicted Class: 4
Confidence: 0.11
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Predicted Class: 15
Confidence: 0.24
